# 03 — Data Exploration

Explore the **processed** Sentinel-2 + USDA CDL inputs that feed the pipeline
(study area: Sacramento Valley, 2024; 10 m grid).

Covers: available S2 acquisition dates, an RGB composite, the CDL label map,
per-crop class distribution, and mean-NDVI temporal profiles per crop.

> Requires processed data locally under `data/processed/` (`stages/fetch_data.py` or `pipeline.py --stages fetch`).

In [ ]:
# Make the pipeline importable as `crop_mapping_pipeline` regardless of the
# checkout directory name (this repo is `cropmap-remote-sensing-exps`; the
# GPU deploy dir is `crop_mapping_pipeline`). Also silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('S2 bands/date:', C.S2_BAND_NAMES)
print('S2 train dir :', C.S2_TRAIN_DIR)
print('CDL train    :', C.CDL_TRAIN)

In [ ]:
import glob, re
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

## 1. Available S2 acquisition dates

One processed multi-band TIF per date (10 land bands each).

In [ ]:
s2_files = sorted(glob.glob(str(C.S2_TRAIN_DIR / '*.tif')))
def parse_date(p):
    m = re.search(r'(\d{4})[_-]?(\d{2})[_-]?(\d{2})', Path(p).name)
    return '-'.join(m.groups()) if m else Path(p).stem
dates = [parse_date(p) for p in s2_files]
print(f'{len(s2_files)} S2 date files in {C.S2_TRAIN_DIR}')
for p, d in zip(s2_files, dates):
    print(f'  {d}   {Path(p).name}')
assert s2_files, 'No S2 files found — fetch processed data first.'
with rasterio.open(s2_files[0]) as src:
    print('\nExample raster:', src.count, 'bands', src.width, 'x', src.height,
          '| CRS', src.crs, '| dtype', src.dtypes[0])

## 2. RGB composite (B4/B3/B2)

Percentile-stretched true-colour view of one date.

In [ ]:
def band_idx(name): return C.S2_BAND_NAMES.index(name)
def stretch(a, lo=2, hi=98):
    a = a.astype(np.float32); a[a == C.S2_NODATA] = np.nan
    p_lo, p_hi = np.nanpercentile(a, [lo, hi])
    return np.clip((a - p_lo) / max(p_hi - p_lo, 1e-6), 0, 1)

date_i = min(len(s2_files) // 2, len(s2_files) - 1)   # a mid-year date
with rasterio.open(s2_files[date_i]) as src:
    r = src.read(band_idx('B4') + 1); g = src.read(band_idx('B3') + 1); b = src.read(band_idx('B2') + 1)
rgb = np.dstack([stretch(r), stretch(g), stretch(b)])
plt.figure(figsize=(8, 8)); plt.imshow(rgb)
plt.title(f'RGB composite — {dates[date_i]}'); plt.axis('off'); plt.show()

## 3. CDL label map

CDL remapped to model indices (0 = background, 1–8 = crops).

In [ ]:
gt, cdl_prof = None, None
with rasterio.open(C.CDL_TRAIN) as src:
    cdl = src.read(1).astype(np.int32)
gt = C.REMAP_LUT[np.clip(cdl, 0, 255)]

# build a colormap: bg=lightgrey then a distinct colour per crop
crop_ids = list(C.CDL_CLASS_NAMES.keys())
palette = plt.cm.tab10(np.linspace(0, 1, len(crop_ids)))
cmap = ListedColormap([(0.9, 0.9, 0.9, 1)] + [tuple(c) for c in palette])
norm = BoundaryNorm(np.arange(-0.5, C.NUM_CLASSES + 0.5), C.NUM_CLASSES)
plt.figure(figsize=(8, 8))
plt.imshow(gt, cmap=cmap, norm=norm, interpolation='nearest')
legend = [Patch(facecolor='0.9', label='Background')] + [
    Patch(facecolor=palette[i], label=name)
    for i, name in enumerate(C.CDL_CLASS_NAMES.values())]
plt.legend(handles=legend, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.title('CDL label map (2024)'); plt.axis('off'); plt.tight_layout(); plt.show()

## 4. Per-crop class distribution

Pixel count / area per crop in the study area.

In [ ]:
px = 10.0  # ~10 m pixel
rows = []
for cdl_id, name in C.CDL_CLASS_NAMES.items():
    n = int((cdl == cdl_id).sum())
    rows.append((name, n, n * px * px / 1e4))   # ha
rows.sort(key=lambda r: -r[1])
names = [r[0] for r in rows]; ha = [r[2] for r in rows]
print(f'{"Crop":<14}{"pixels":>12}{"area (ha)":>12}')
for nm, n, h in rows: print(f'{nm:<14}{n:>12,}{h:>12,.0f}')
plt.figure(figsize=(9, 4)); plt.bar(names, ha, color='seagreen')
plt.ylabel('Area (ha)'); plt.title('CDL crop area — study area (2024)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 5. Mean-NDVI temporal profile per crop

Samples labelled pixels per crop and plots the mean NDVI across all dates —
the phenological signal the multi-temporal / selection scenarios exploit.

In [ ]:
rng = np.random.default_rng(C.SEED)
SAMPLE = 2000   # pixels per crop
nir_i, red_i = band_idx('B8'), band_idx('B4')
# pick pixel coords per crop
coords = {}
for cdl_id, name in C.CDL_CLASS_NAMES.items():
    ys, xs = np.where(cdl == cdl_id)
    if len(ys) == 0: continue
    sel = rng.choice(len(ys), min(SAMPLE, len(ys)), replace=False)
    coords[name] = (ys[sel], xs[sel])

ndvi_series = {name: [] for name in coords}
for p in s2_files:
    with rasterio.open(p) as src:
        nir = src.read(nir_i + 1).astype(np.float32)
        red = src.read(red_i + 1).astype(np.float32)
    for arr in (nir, red):
        arr[arr == C.S2_NODATA] = np.nan
    ndvi = (nir - red) / (nir + red + 1e-6)
    for name, (ys, xs) in coords.items():
        ndvi_series[name].append(np.nanmean(ndvi[ys, xs]))

plt.figure(figsize=(11, 5))
for name, series in ndvi_series.items():
    plt.plot(dates, series, marker='o', ms=3, label=name)
plt.ylabel('Mean NDVI'); plt.title('Per-crop NDVI temporal profile (2024)')
plt.xticks(rotation=60, ha='right', fontsize=7)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()